# 00_setup_y_validacion_api\n
\n
## Objetivo\n
- Verificar entorno de trabajo y dependencias.\n
- Cargar credenciales desde `.env`.\n
- Confirmar autenticacion autorizada con API oficial de X.\n
\n
## Entradas esperadas\n
- Archivo `.env` local con credenciales validas.\n
\n
## Salidas esperadas\n
- Registro de validacion de entorno y acceso API.\n
\n
## Nota\n
Notebook de preparacion. Sin implementacion de recoleccion en esta etapa.\n

In [ ]:
#vamos a validar el token

import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nWorking directory:")
print(Path.cwd())

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print(".env exists:", (PROJECT_ROOT / ".env").exists())

In [ ]:
import sys

!{sys.executable} -m pip install -U python-dotenv requests pandas pyyaml tqdm

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

bearer_token = os.getenv("X_BEARER_TOKEN")

if not bearer_token:
    raise ValueError("No se encontró X_BEARER_TOKEN en .env")

print("Token cargado correctamente.")
print("Primeros caracteres:", bearer_token[:8] + "...")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {bearer_token}"
}

url = "https://api.x.com/2/users/by/username/XDevelopers"

params = {
    "user.fields": "id,name,username,verified"
}

response = requests.get(url, headers=headers, params=params, timeout=30)

print("Status code:", response.status_code)
print(response.text[:1000])

In [ ]:
from datetime import datetime, timezone
import requests

url = "https://api.x.com/2/tweets/counts/all"

params = {
    "query": 'from:XDevelopers lang:en -is:retweet',
    "start_time": "2026-01-01T00:00:00Z",
    "end_time": "2026-01-02T00:00:00Z",
    "granularity": "day",
}

response = requests.get(url, headers=headers, params=params, timeout=30)

print("Status code:", response.status_code)
print(response.text[:2000])

In [ ]:
url = "https://api.x.com/2/tweets/search/all"

params = {
    "query": 'from:XDevelopers lang:en -is:retweet',
    "start_time": "2026-01-01T00:00:00Z",
    "end_time": "2026-01-02T00:00:00Z",
    "max_results": 10,
    "tweet.fields": "id,text,created_at,author_id,conversation_id,public_metrics,lang",
}

response = requests.get(url, headers=headers, params=params, timeout=30)

print("Status code:", response.status_code)
print(response.text[:2000])

In [ ]:
import pandas as pd
from datetime import datetime, timezone

validation_report = pd.DataFrame([
    {
        "check": "env_file_exists",
        "status": (PROJECT_ROOT / ".env").exists(),
        "details": str(PROJECT_ROOT / ".env"),
    },
    {
        "check": "bearer_token_loaded",
        "status": bool(bearer_token),
        "details": "Token cargado sin imprimir completo",
    },
    {
        "check": "user_lookup_status",
        "status": "see_notebook_output",
        "details": "Endpoint /2/users/by/username/XDevelopers",
    },
    {
        "check": "full_archive_counts_status",
        "status": "see_notebook_output",
        "details": "Endpoint /2/tweets/counts/all",
    },
    {
        "check": "full_archive_search_status",
        "status": "see_notebook_output",
        "details": "Endpoint /2/tweets/search/all",
    },
    {
        "check": "validated_at",
        "status": True,
        "details": datetime.now(timezone.utc).isoformat(),
    },
])

output_path = PROJECT_ROOT / "outputs" / "tables" / "api_validation_report.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
validation_report.to_csv(output_path, index=False)

validation_report